In [ ]:
import os
import logging

from dotenv import load_dotenv

from loading.DocumentLoaderManager import DocumentLoaderManager
from chunking.chunking import Chunking
from embedding.embedding import Embedding
from query_transform.query_transform import QueryTransform
from reranking.reranking import Reranker
from answer_generation.answer_generation import AnswerGeneration
from evaluation.rag_evaluation import RAGEvaluator


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logger = logging.getLogger(__name__)


load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Document-RAG-Project"

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing")

if not os.getenv("LANGSMITH_API_KEY"):
    raise ValueError("LANGSMITH_API_KEY is missing")



In [ ]:
class RAGApplication:

    def __init__(self, file_path):

        self.file_path = file_path

        self.documents = []
        self.chunks = []
        self.vectorstore = None
        self.document_loader = DocumentLoaderManager()
        self.embedding_manager = Embedding()
        self.query_transformer = QueryTransform()
        self.reranker = Reranker()
        self.answer_generator = AnswerGeneration()

        logger.info("RAG application initialized")


    def load_documents(self):

        self.documents = (
            self.document_loader.initialize_document(
                self.file_path
            )
        )

        if not self.documents:
            raise ValueError("Document loading failed")

        logger.info(
            "Loaded %d documents",
            len(self.documents)
        )

        return self.documents

    def select_strategy(self):

        strategy = self.document_loader.check_toc()

        logger.info("Selected RAG strategy: %s", strategy)

        return strategy

    def create_chunks(self):

        chunker = Chunking(self.documents)

        self.chunks = chunker.text_split()

        if not self.chunks:
            raise ValueError("Chunk creation failed")

        logger.info(
            "Created %d chunks",
            len(self.chunks)
        )

        return self.chunks

    def create_vectorstore(self):

        self.vectorstore = (
            self.embedding_manager.create_embed_vectorstore(
                self.chunks,
                batch_size=64
            )
        )

        self.embedding_manager.save()

        logger.info("FAISS vector store created and saved")

        return self.vectorstore

    def rewrite_query(self, question):

        rewritten_query = (
            self.query_transformer.rewrite_query(question)
        )

        logger.info("Query rewritten successfully")

        return rewritten_query

    def retrieve_and_rerank(
        self,
        query,
        candidate_k=20,
        final_k=5
    ):

        candidates = self.embedding_manager.search(
            query=query,
            k=candidate_k
        )

        ranked_results = self.reranker.rerank(
            query=query,
            documents=candidates,
            top_n=final_k
        )

        logger.info(
            "Retrieved %d candidates and reranked to %d",
            len(candidates),
            len(ranked_results)
        )

        return ranked_results

    def generate_answer(self, question, results):

        answer = self.answer_generator.generate_answer(
            question=question,
            reranked_results=results
        )

        return answer

    def run_pipeline(self, question):

        if self.vectorstore is None:
            raise RuntimeError(
                "Build the vector store before asking questions"
            )

        try:

            logger.info("Processing question: %s", question)

    
            rewritten_query = self.rewrite_query(question)

            
            results = self.retrieve_and_rerank(
                query=rewritten_query,
                candidate_k=20,
                final_k=5
            )

            answer = self.generate_answer(
                question=question,
                results=results
            )

            context = "\n\n".join(
                item["document"].page_content
                for item in results
            )

            return {
                "answer": answer,
                "context": context,
                "rewritten_query": rewritten_query,
                "retrieved_documents": [
                    item["document"].page_content
                    for item in results
                ],
                "reranking_scores": [
                    item["score"]
                    for item in results
                ]
            }

        except Exception:
            logger.exception("RAG pipeline failed")
            raise

    def build(self):

        logger.info("Building RAG pipeline")

        self.load_documents()

        strategy = self.select_strategy()

        self.create_chunks()

        self.create_vectorstore()

        logger.info("RAG pipeline ready")

        return strategy


    def evaluate(self, dataset_name):

        evaluator = RAGEvaluator(
            dataset_name=dataset_name
        )

        return evaluator.evaluate(self.run_pipeline)



if __name__ == "__main__":

    FILE_PATH = "data/document.pdf"

    app = RAGApplication(FILE_PATH)
    strategy = app.build()

    print("\nSelected RAG Strategy:", strategy)

    question = input("\nEnter your question: ")

    result = app.run_pipeline(question)

    print("\nRewritten Query:")
    print(result["rewritten_query"])

    print("\nFinal Answer:")
    print(result["answer"])


    RUN_EVALUATION = False

    if RUN_EVALUATION:

        evaluation_results = app.evaluate(
            dataset_name="Document-RAG-Test-Dataset"
        )

        print("\nLangSmith Evaluation Results:")
        print(evaluation_results)
